# AKRM v2 — Stage 1: Heuristic Policy Validation

**Research Question:** Can a knowledge-guided heuristic policy outperform random intervention selection under identical learning conditions?

**Experimental Design:**
- **Control Group:** Random provider selection (`--akrm_policy random`)
- **Test Group:** Knowledge-guided heuristic planner (`--akrm_policy heuristic`)
- **Controlled Variables:** Same model, dataset, seeds, training schedule, providers, executor

> Run all cells **in order**. Cell 2 (Random) must finish before Cell 3 (Heuristic).

## Cell 1 — Setup: Clone Repository & Install Dependencies

In [1]:
!git clone https://github.com/Captdumbledore/Uncertainty-driven-learning-framework_MVP.git
%cd Uncertainty-driven-learning-framework_MVP
!pip install -q -r requirements.txt
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Cloning into 'Uncertainty-driven-learning-framework_MVP'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 48 (delta 13), reused 45 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 53.58 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/Uncertainty-driven-learning-framework_MVP
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Cell 2 — Control Group: Random Provider Selection

This is the **baseline experiment**. The AKRM planner randomly selects between Retrieval and Counterexample providers, ignoring the diagnosed knowledge gap entirely.

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy random \
    --output_dir outputs_cifar10_random


  UNCERTAINTY-GUIDED RETRAINING — RESEARCH PROTOTYPE
  Dataset          : CIFAR10
  Initial epochs   : 15
  Retraining epochs: 8
  MC samples       : 30
  Top fraction     : 10%
  Seeds            : [42, 123, 999]
  Device           : cuda
  Output dir       : outputs_cifar10_random/

  SEED: 42

  --------------------------------------------------------------
  Phase 1 — Loading data and training baseline CNN
  --------------------------------------------------------------
  1% 2.39M/170M [01:10<1:24:41, 33.1kB/s]

## Cell 3 — Test Group: Knowledge-Guided Heuristic Planner

This is the **test experiment**. The AKRM planner selects a provider based on the diagnosed knowledge gap type, entropy, and decision margin.

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --output_dir outputs_cifar10_heuristic

## Cell 4 — Results Comparison

Read and compare the aggregated results from both runs.

In [ ]:
import json, os

def load_results(output_dir):
    path = os.path.join(output_dir, 'aggregated_results.json')
    if not os.path.exists(path):
        print(f'Not found: {path}')
        return None
    with open(path) as f:
        return json.load(f)

random_results    = load_results('outputs_cifar10_random')
heuristic_results = load_results('outputs_cifar10_heuristic')

print('\n' + '='*60)
print('STAGE 1 RESULTS — HEURISTIC vs RANDOM PROVIDER SELECTION')
print('='*60)

for label, results in [('RANDOM', random_results), ('HEURISTIC', heuristic_results)]:
    if results is None:
        continue
    print(f'\n--- {label} Policy ---')
    for pipeline, metrics in results.items():
        if isinstance(metrics, dict) and 'accuracy' in metrics:
            print(f"  Pipeline {pipeline}: Acc={metrics['accuracy']:.4f}  F1={metrics.get('f1',0):.4f}  ECE={metrics.get('ece',0):.4f}")